# VAC-WTMetaD simulation using "data rich" CVs


# 1. Grab SRV from unbiased simulation folder

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import openmm as mm
from openmm.app import Simulation, ForceField
from openmmtorch import TorchForce

from src.data import simulation_data

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using CUDA, Torch version {torch.__version__}')
    print('-------------------------------------')
else:
    device = torch.device('cpu')
dtype = torch.float32

# sd = simulation_data('data/unbiased', assign_labels=['srv_net'])
# net = sd.srv_net.cpu()

/home/bharland/anaconda3/envs/openmm-torch/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1752524775208/work/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
torch.cuda.is_available()

False

# 2. Simulation

#### TODO:  add bias force force_group_id line of code and output this value to file.  See `src.util.bias_from_context`

```python
step = n * p.steps_per_gaussian 
if step ..:
    bias = bias_from_context(simulation, force_group_id)
    biases.append(bias)
```

In [ ]:
from src.param import SimulationParametersMetaD
from src.metad import SimulationCVS, metadynamics, kde_compression
from src.util import create_system, state_data_reporter, hdf5_reporter
from tqdm.notebook import tqdm

p = SimulationParametersMetaD(working_dir='data/datarich-metad')
scvs = SimulationCVS(p.pdb_file)
pdb = scvs.pdb()
forcefield = ForceField('amber99sb.xml', 'tip3p.xml')

do_simulation = False
if do_simulation:
    num_sims = 100
    sim_start = 15 # -> 22 : Apr. 21, 2025

    for sim_num in range(sim_start, num_sims):
        print(f'starting simulation {sim_num}')
        if sim_num == 0:
            metad = metadynamics(p.temperature, p.bias_factor, p.height, p.width)
            metad.add_gaussian(scvs.pdb_cvs(net))
            start_positions = pdb.positions
        else:
            if sim_num == sim_start:
                sd = simulation_data(p, subdir=(sim_num - 1))
            metad = sd.metad
            start_positions = sd.final_positions

        sd = simulation_data(p, subdir=sim_num)
        module = metad.force_module(net)
        torch.jit.script(module).save('data/force_module.pt')
        torch_force = TorchForce('data/force_module.pt')
        torch_force.addGlobalParameter('add_gaussian', False)
        torch_force.addGlobalParameter('height', 0.0)
        torch_force.addGlobalParameter('center1', 0.0)
        torch_force.addGlobalParameter('center2', 0.0)

        integrator = mm.LangevinIntegrator(p.temperature, p.friction_coeff, p.timestep)
        system = create_system(forcefield, pdb.topology)
        system.addForce(torch_force)
        # force_group_id = system.addForce(force)

        simulation = Simulation(pdb.topology, system, integrator)
        simulation.context.setPositions(start_positions)
        sdr = state_data_reporter(str(sd.files['outfile']), p.report_interval)
        hdr = hdf5_reporter(str(sd.files['h5file']), p.report_interval)
        simulation.reporters.append(sdr)
        simulation.reporters.append(hdr)
        simulation.step(p.steps_per_gaussian)

        for n in tqdm(range(p.num_gaussians - 1)):
            metad.add_gaussian(scvs.sim_cvs(net, simulation))
            simulation.context.setParameter('add_gaussian', True)
            simulation.context.setParameter('height', metad.heights[-1])
            simulation.context.setParameter('center1', metad.centers[-1][0])
            simulation.context.setParameter('center2', metad.centers[-1][1])
            simulation.step(1)
            simulation.context.setParameter('add_gaussian', False)
            simulation.step(p.steps_per_gaussian - 1)

        metad_kde = kde_compression(metad, dist_threshold=1)
        print(f'compressing metad: {len(metad)} -> {len(metad_kde)} kernels\n')
        r = simulation.context.getState(getPositions=True).getPositions()

        sd.save_and_assign_objects({'final_positions': r,
                                    'metad': metad,
                                    'metad_kde': metad_kde})

        # h5 file must be closed before 'sd.save_feature_data'
        for reporter in simulation.reporters:
            if hasattr(reporter, 'close'):
                reporter.close()

        sd.save_feature_data(recalculate=True, pdbfile=p.pdb_file)

# 3. Gather data and compute biases
### A. `features`, `dihedrals`, `cvs` and other eigen data

In [45]:
num_sims = 22
features = []
dihedrals = []

for step in range(num_sims):
    sd = simulation_data(p, subdir=step, loud=False)
    features.append(sd.features)
    dihedrals.append(sd.dihedrals)

# Final metad object contains every Gaussian kernel from full simulation
metad = sd.metad

sd = simulation_data(p)

do_calc = False
if do_calc:
    sd.save_and_assign_objects({'features': np.vstack(features),
                                'dihedrals': np.vstack(dihedrals)})
    srv = simulation_data('data/unbiased', assign_labels=['srv']).srv
    sd.save_eigen_data(srv)

Loaded SimulationData object from data/datarich-metad with 21978 frames


In [ ]:
def num_gaussians_frame(p, frame):
    """Return the number of Gaussians that made up the MetaD potential at the time that the CVs in frame 'frame' were recorded

    Careful about how Metadynamics objects actually construct bias potential!

    Each step has:
        # frames = p.num_frames
        # Gaussian kernels = p.num_gaussians

    Within curent step:
        (i) how many Gaussian kernels per frame?
        (ii) take floor of frame_in_step * gaussians_per_frame
    """
    step = frame // p.num_frames
    gaussians_to_step = step * p.num_gaussians

    frame_in_step = frame % p.num_frames
    gaussians_per_frame = p.num_steps_per_frame / p.steps_per_gaussian
    gaussian_in_step = int(np.floor((frame_in_step + 1) * gaussians_per_frame))

    return gaussians_to_step + gaussian_in_step

biases = []
print(f'calculating biases')
for frame, s in tqdm(enumerate(sd.cvs)):
    ng = num_gaussians_frame(p, frame)
    bias = metad.bias_potential(s, num_gaussians=ng)
    biases.append(metad.bias_potential(s, num_gaussians=ng))

sd.save_and_assign_objects({'biases': np.array(biases)})

calculating biases


0it [00:00, ?it/s]

In [47]:
len(sd.cvs)

21978

In [48]:
len(sd.biases)

21978

In [49]:
sd.biases[:100]

array([0.00000000e+00, 6.75850662e+00, 9.76436807e+00, 1.01925549e+01,
       7.21461078e-34, 2.31105696e-07, 2.61163789e-01, 4.25563221e-01,
       5.63626352e+00, 1.01718282e+01, 1.32646244e+01, 5.57358553e+00,
       6.96912227e+00, 1.47190826e+01, 1.49344115e+01, 1.74843411e+01,
       1.86378458e+01, 2.05760471e+01, 2.09305752e+01, 6.97849373e+00,
       6.84687668e+00, 3.78196001e+00, 7.15784077e+00, 7.34350994e+00,
       7.98515354e+00, 1.69266014e+01, 1.74914022e+01, 1.86798549e+01,
       1.60379258e-34, 0.00000000e+00, 5.00318966e-09, 2.46854806e-01,
       6.16508549e-01, 2.54914435e+00, 2.87601275e+00, 2.75614202e+00,
       2.13981713e+00, 8.98865683e-01, 1.87974042e+00, 4.18902819e+00,
       3.09431413e+00, 9.83586299e+00, 7.23092782e+00, 1.47017150e+01,
       1.67095665e+01, 1.87920797e+01, 1.28430944e-05, 5.81980752e-41,
       7.45248435e+00, 1.97477028e+01, 2.26810066e+01, 2.38150810e+01,
       1.21409557e+01, 1.02687066e-10, 6.49647761e-03, 4.76142144e+00,
      